# Advanced Solar Activity Forecasting: Mathematical Framework
### Integration of Hathaway Parametrics, Adaptive Kalman Filtering, and Dynamic Mode Decomposition (DMD)

This report formalizes the predictive engine used for sunspot number (SSN) and latitudinal migration analysis. We combine physical heuristics with data-driven operator theory to provide robust forecasts for Solar Cycle 25 and beyond.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Publication Style Configuration
plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#334155',
    'grid.alpha': 0.15,
    'font.family': 'serif',
    'legend.frameon': True,
    'legend.edgecolor': '#e2e8f0'
})

COLORS = {'obs': '#64748b', 'kalman': '#10b981', 'dmd': '#8b5cf6', 'future_bg': '#f0fdf4'}

## 1. Discretization and Vectorization via Latitudinal Binning

To analyze the Spörer diagram mathematically, we transform individual sunspot observations into a time-series of state vectors. Given the heliographic latitude $\theta \in [-50^\circ, 50^\circ]$, we define a discretization into $M$ bins of size $\Delta \theta$:

$$V_j(t) = \sum_{i=1}^{N_{spots}(t)} \mathbb{1}_{(\theta_j \leq \theta_i < \theta_{j+1})}$$

The complete state of the solar disk at time $t$ is represented by the vector $\mathbf{x}_t = [V_1(t), V_2(t), \dots, V_M(t)]^T \in \mathbb{R}^M$. The collection of $N$ monthly observations forms the snapshot matrix:

$$\mathbf{X} = [\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_N] \in \mathbb{R}^{M \times N}$$

In [ ]:
path = r'labels.csv'
df = pd.read_csv(path)
df['date'] = pd.to_datetime(df['date_iso'])
df['year_month'] = df['date'].dt.to_period('M')

monthly_hist = df.groupby('year_month').size().reset_index(name='ssn')
monthly_hist['year_float'] = monthly_hist['year_month'].apply(lambda x: x.year + (x.month - 1) / 12)

lat_bins = np.arange(-50, 51, 4)
snapshots = np.zeros((len(lat_bins)-1, len(monthly_hist)))
for j, t in enumerate(monthly_hist['year_month']):
    snapshots[:, j], _ = np.histogram(df[df['year_month'] == t]['lat'], bins=lat_bins)

## 2. Dynamic Mode Decomposition (DMD) Formalism

DMD seeks the best linear operator $\mathbf{A}$ that maps a snapshot to the next:

$$\mathbf{x}_{k+1} \approx \mathbf{A} \mathbf{x}_k$$

Defining $\mathbf{X}_1 = [\mathbf{x}_1, \dots, \mathbf{x}_{N-1}]$ and $\mathbf{X}_2 = [\mathbf{x}_2, \dots, \mathbf{x}_N]$, we solve for $\mathbf{A}$ using a truncated Singular Value Decomposition (SVD) of $\mathbf{X}_1 = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^H$:

$$\tilde{\mathbf{A}} = \mathbf{U}_r^H \mathbf{X}_2 \mathbf{V}_r \boldsymbol{\Sigma}_r^{-1}$$

The latitudinal migration is forecasted by iteratively applying $\mathbf{A}$. For historical adjustment, we use the one-step ahead predictor:

$$\hat{\mathbf{x}}_k = \mathbf{A} \mathbf{x}_{k-1}$$

In [ ]:
X1, X2 = snapshots[:, :-1], snapshots[:, 1:]
U, S, Vh = svd(X1, full_matrices=False)
r = 10
Ur, Sr, Vr = U[:, :r], np.diag(S[:r]), Vh[:r, :].T
A = X2 @ Vr @ np.linalg.inv(Sr) @ Ur.T

HORIZON = 60
dmd_ssn_raw, butterfly_dots = [], []

for t in range(len(monthly_hist)):
    pred_v = snapshots[:, 0] if t == 0 else A @ snapshots[:, t-1]
    pred_v = np.maximum(0, pred_v)
    dmd_ssn_raw.append(np.sum(pred_v))
    if t % 2 == 0:
        for l in range(len(lat_bins)-1):
            if pred_v[l] > 0.5:
                butterfly_dots.append({'y': monthly_hist['year_float'].iloc[t], 'l': lat_bins[l]+2, 's': pred_v[l], 'is_f': False})

curr_v = snapshots[:, -1]
future_yf = [monthly_hist['year_float'].max() + (i+1)/12 for i in range(HORIZON)]
for t in range(HORIZON):
    curr_v = np.maximum(0, (A @ curr_v) * 0.98)
    dmd_ssn_raw.append(np.sum(curr_v))
    if t % 2 == 0:
        for l in range(len(lat_bins)-1):
            if curr_v[l] > 0.5:
                butterfly_dots.append({'y': future_yf[t], 'l': lat_bins[l]+2, 's': curr_v[l], 'is_f': True})

butterfly_df = pd.DataFrame(butterfly_dots)

## 3. Discrete Integral and Scaling (DMD SSN)

The total activity (SSN) derived from DMD is obtained via a discrete latitudinal integral:

$$SSN_{DMD}(t) = \kappa \cdot \sum_{j=1}^M \hat{V}_j(t)$$

The scaling factor $\kappa$ is computed to minimize the bias against historical observations:

$$\kappa = \frac{\sum_{t=1}^N SSN_{obs}(t)}{\sum_{t=1}^N \left( \sum_{j=1}^M \hat{V}_j(t) \right)}$$

In [ ]:
kappa = np.sum(monthly_hist['ssn']) / np.sum(dmd_ssn_raw[:len(monthly_hist)])
dmd_ssn = np.array(dmd_ssn_raw) * kappa

## 4. Podladchikova + Kalman Filter Formalism

We model the SSN series using an adaptive Kalman Filter. The state transition is guided by the **Hathaway Physical Model** $R_H(v)$:

**1. Prediction Stage:**
$$x_{k|k-1} = \Phi_k x_{k-1|k-1}, \quad \Phi_k = \frac{R_H(v_k)}{R_H(v_{k-1})}$$
$$P_{k|k-1} = \Phi_k^2 P_{k-1|k-1} + Q$$

**2. Update Stage (with observation $z_k$):**
$$K_k = \frac{P_{k|k-1}}{P_{k|k-1} + R}$$
$$x_{k|k} = x_{k|k-1} + K_k (z_k - x_{k|k-1})$$
$$P_{k|k} = (1 - K_k) P_{k|k-1}$$

Where $Q$ and $R$ represent the process and observation noise covariance respectively.

In [ ]:
def hathaway(v, a, b): return (a * (v**3)) / (np.exp((v*v)/(b*b)) - 0.71) if v > 0 else 0.0
a_p, target = 0.01, 140
for _ in range(10): 
    b_p = 27.12 + 25.15 / np.power(a_p * 1000, 0.25)
    a_p *= (target / max(1, hathaway(50, a_p, b_p)))
b_p = 27.12 + 25.15 / np.power(a_p * 1000, 0.25)

full_yf = np.concatenate([monthly_hist['year_float'].values, future_yf])
is_f = np.concatenate([np.zeros(len(monthly_hist)), np.ones(HORIZON)]).astype(bool)
def get_v(yf): return max(1, int(round((yf - (2019.9 if yf >= 2019.9 else 2008.9)) * 12)))

kalman_ssn, state_r, state_p = [], monthly_hist['ssn'].iloc[0], 10.0
for yf in full_yf:
    phi = hathaway(get_v(yf), a_p, b_p) / max(0.1, hathaway(get_v(yf - 1/12), a_p, b_p))
    r_pred = phi * state_r
    state_p = phi**2 * state_p + 0.2 * r_pred
    idx = np.where(monthly_hist['year_float'] == yf)[0]
    if len(idx) > 0:
        k_g = state_p / (state_p + 2.5 * r_pred)
        state_r = r_pred + k_g * (monthly_hist['ssn'].iloc[idx[0]] - r_pred)
        state_p = (1 - k_g) * state_p
    else: state_r = r_pred
    kalman_ssn.append(state_r)

## 5. Temporal Evolution Analysis (SSN Projection)

The figure below illustrates the synchronization between the Adaptive Kalman Filter and the DMD latitudinal integral.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(monthly_hist['year_float'], monthly_hist['ssn'], color=COLORS['obs'], alpha=0.1, lw=8, label='Historical Observation')
plt.plot(full_yf[~is_f], np.array(kalman_ssn)[~is_f], color=COLORS['kalman'], lw=2, label='Adaptive Kalman Filter')
plt.plot(full_yf[is_f], np.array(kalman_ssn)[is_f], color=COLORS['kalman'], lw=2, ls='--')
plt.plot(full_yf[~is_f], dmd_ssn[~is_f], color=COLORS['dmd'], lw=1.5, alpha=0.7, label='DMD Spörer Integral')
plt.plot(full_yf[is_f], dmd_ssn[is_f], color=COLORS['dmd'], lw=1.5, ls='--', alpha=0.7)

plt.axvline(monthly_hist['year_float'].max(), color='#64748b', ls=':', lw=1.5)
plt.axvspan(2019.9, full_yf[-1], color=COLORS['future_bg'], alpha=0.3, zorder=-1)
plt.title('SUNSPOT NUMBER (SSN) PROJECTION LAYER', loc='left', fontweight='bold', pad=20)
plt.ylabel('SSN Count', fontweight='bold')
plt.xlabel('Observation Year', fontweight='bold')
plt.xlim(2010, 2030); plt.ylim(0, 250)
plt.legend(loc='upper right', frameon=True)
plt.savefig('figure_ssn_projection.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Spörer Diagram: Latitudinal Migration Analysis

The Butterfly diagram demonstrates the descent of sunspot production towards the solar equator, modeled via the Koopman Operator.

In [ ]:
plt.figure(figsize=(14, 6))
plt.scatter(df['date'].dt.year + df['date'].dt.month/12, df['lat'], s=0.5, color=COLORS['obs'], alpha=0.03)
adj = butterfly_df[~butterfly_df['is_f']]
fct = butterfly_df[butterfly_df['is_f']]
plt.scatter(adj['y'], adj['l'], s=adj['s']*1.5, color=COLORS['dmd'], alpha=0.2, label='DMD Adjustment')
plt.scatter(fct['y'], fct['l'], s=fct['s']*1.5, color=COLORS['dmd'], alpha=0.8, label='DMD Forecast')

plt.axvline(monthly_hist['year_float'].max(), color='#64748b', ls=':', lw=1.5)
plt.axvspan(2019.9, full_yf[-1], color=COLORS['future_bg'], alpha=0.3, zorder=-1)
plt.axhline(0, color='black', lw=0.5, alpha=0.2)
plt.title('SPÖRER DIAGRAM: LATITUDINAL RECONSTRUCTION', loc='left', fontweight='bold', pad=20)
plt.ylabel('Latitude (°)', fontweight='bold')
plt.xlabel('Year', fontweight='bold')
plt.xlim(2010, 2030); plt.ylim(-50, 50)
plt.legend(loc='lower left', markerscale=4, frameon=True)
plt.savefig('figure_sporer_migration.png', dpi=300, bbox_inches='tight')
plt.show()